---
# Seminar 3 - Grundlagen Ablaufplanung
---





---
### Einlesen der Klassen aus erster Veranstaltung 
---

In [2]:
# files from first session
from InputData import InputData
from OutputData import OutputJob
# Additionally
import numpy


---
### 1. Codierung und Bewertung einer Lösung
---
Im Rahmen Ihrer Werkstudententätigkeit bei Dr. Best sollen Sie die Produktionsleiterin Traudel Teufel bei der Ablaufplanung unterstützen. Aktuell wird die Fertigungsreihenfolge für die nächste Woche immer freitags festgelegt. Aufgrund der zunehmenden Komplexität fällt es Frau Teufel immer schwerer, gute Ablaufpläne zu generieren. Derzeit nutzt sie Ihre lange Erfahrung im Bereich der Zahnpastafertigung und erzeugt manuell eine geeignete Bearbeitungsreihenfolge, welche Sie an die Schichtleiter weitergibt.

In der nächsten Woche sollen 11 Aufträge abgearbeitet werden. Die Inputdaten sind Ihnen in der Datei "InputFlowshopSIST.json" gegeben. Traudel Teufel plant mit folgender Reihenfolge:
**6-5-7-4-8-3-9-2-10-1-11**

Diese Auftragsreihenfolge soll auf allen Maschinen eingehalten werden (Permutation Flow Shop). Leider kann die Produktionsleiterin nicht einschätzen, wie gut Ihre Lösung tatsächlich ist. Da sie von Ihren Programmierfähigkeiten gehört hat, bittet sie Sie, diese Lösung zu bewerten. Anschließend sollen Sie außerdem Auskunft darüber geben, wann welcher Auftrag auf welcher Maschine laut Plan bearbeitet wird.

#### a.) 
Sie überlegen, dass es sinnvoll ist, eine eigene Klasse für Lösungen anzulegen, wenn Sie im folgenden verschiedene Lösungen erzeugen wollen. Deshalb schreiben Sie als erstes eine Klasse **Solution**, welche ein Dictionary mit allen Aufträge als **OutputJobs** sowie die Fertigungsreihenfolge als Liste **Permutation** enthält. Zudem sollen im Konstruktor die Attribute Makespan, TotalTardiness und TotalWeightedTardiness angelegt und zunächst auf -1 gesetzt werden. Nachdem Sie die Klasse definiert haben, bietet es sich an, als erstes Objekt dieser Klasse die von Frau Teufel vorgeschlagene Lösung mit Hilfe der Inputdaten in der Datei "InputFlowshopSIST.json" zu erzeugen. Nennen Sie diese erste Lösung **DevilSolution**.


In [3]:
class Solution:
    def __init__(self, jobs, permutation):
        self.permutation = permutation
        
        self.makespan = -1
        self.total_tardiness = -1
        self.total_weighted_tardiness = -1

        self.OutputJobs = {}
        for jobid, job in enumerate(jobs):
            self.OutputJobs[jobid] = OutputJob(job)

    def __str__(self):
        return f'The permutation {self.permutation} results in a Makespan of {self.makespan}'
    
    def setPermutation(self, Permutation):
        self.permutation = Permutation

#### Führen Sie im Anschluss folgenden Code aus ####
data = InputData("../../../data/Prescriptive_Lessons/InputFlowshopSIST.json")
Permutation = [x-1 for x in [6,5,7,4,8,3,9,2,10,1,11]]
DevilSolution = Solution(data.InputJobs, Permutation)
print(DevilSolution)

The permutation [5, 4, 6, 3, 7, 2, 8, 1, 9, 0, 10] results in a Makespan of -1


In [ ]:
# repition myself of the Solution class
class Solution:
    def __init__(self, jobdata, permutation):
        self. permutation = permutation  # List of job IDs in the order they are processed
        self.makespan = -1
        self.total_tardiness = -1
        self.total_weighted_tardiness = -1

        self.OutputJobs = {}
        for id, data in enumerate(jobdata):
            self.OutputJobs[id] = OutputJob(data)

    def __str__(self):
        return f'The permutation {self.permutation} results in a Makespan of {self.makespan}'
    
    def setPermutation(self, Permutation):
        self.permutation = Permutation
    
flowshop_data = InputData('../../../data/Prescriptive_Lessons/InputFlowshopSIST.json')
permutation = [x-1 for x in [6,5,7,4,8,3,9,2,10,1,11]]
DevilSolution = Solution(flowshop_data.InputJobs, permutation)

#### b.) 
Um die Qualität einer Lösung einschätzen zu können, sollten Sie als nächstes eine Bewertungsfunktion für ein gegebenes Solution Objekt schreiben. Diese Methode sollte für eine gegebene Reihenfolge (Permutation) allen Aufträgen Start- und Endzeitpunkte zuweisen unter Beachtung, dass jede Maschine nur einen Auftrag zur selben Zeit bearbeiten kann und ein Auftrag nicht gleichzeitig auf mehreren Maschinen bearbeitet werden kann. Die Rüstzeiten können Sie dabei zunächst vernachlässigen. Am Ende der Einplanung können Sie für das Solution Objekt noch das Attribut Makespan festlegen.

Da die erstellte Funktion zur Bewertung einer Lösung benötigt wird, erachten Sie es als sinnvoll, diese der Klasse EvaluationLogic anzuhängen.

Abschließend können Sie Ihre neu entwickelten Methoden testen, indem Sie die von Frau Teufel vorgeschlagene Lösung **DevilSolution** bewerten.

In [4]:
class EvaluationLogic:
    def DefineStartEnd(self, currentSolution):
        for position, jobid in enumerate(currentSolution.permutation):
            currentjob = currentSolution.OutputJobs[jobid]
            # Start and EndTimes of the first job in the permutation
            # schedule first job: starts when finished at previous stage
            if position == 0:
                currentjob.EndTimes = numpy.cumsum(currentjob.ProcessingTimes).tolist()
                currentjob.StartTimes[1:] = currentjob.EndTimes[:-1]
            
            # Start and EndTimes of the consecutive jobs in the permutation
            # schedule further jobs: starts when finished at previous stage and the predecessor is no longer on the considered machine
            else:
                # first machine
                currentjob.StartTimes[0] = previousjob.EndTimes[0]
                currentjob.EndTimes[0] = currentjob.StartTimes[0] + currentjob.ProcessingTimes[0]
                # all other machines
                for machine in range(1, len(currentjob.ProcessingTimes)):
                    currentjob.StartTimes[machine] = max(previousjob.EndTimes[machine], currentjob.EndTimes[machine-1])
                    currentjob.EndTimes[machine] = currentjob.StartTimes[machine] + currentjob.ProcessingTimes[machine]
            previousjob = currentjob
        
        # Makespan
        # currentSolution.makespan = currentjob.EndTimes[-1]
        currentSolution.makespan = currentSolution.OutputJobs[currentSolution.permutation[-1]].EndTimes[-1]

EvaluationLogic().DefineStartEnd(DevilSolution)
print(DevilSolution)
print(DevilSolution.makespan)

The permutation [5, 4, 6, 3, 7, 2, 8, 1, 9, 0, 10] results in a Makespan of 8922
8922


Erwarteter Output:

    8922

In [5]:
{id+1: DevilSolution.OutputJobs[id].StartTimes for id in DevilSolution.permutation}

{6: [0, 796, 1041, 1673, 2048],
 5: [796, 1324, 1673, 2462, 2586],
 7: [1324, 1856, 2462, 3005, 3901],
 4: [1856, 2316, 3005, 3901, 4353],
 8: [2316, 2858, 3528, 4021, 4852],
 3: [2330, 2982, 3858, 4564, 5637],
 9: [2342, 3858, 4385, 5138, 6402],
 2: [2599, 4385, 5138, 5896, 6865],
 10: [3231, 4837, 5896, 6174, 7263],
 1: [4127, 5733, 6110, 6432, 7522],
 11: [4502, 5745, 6252, 6753, 7934]}

In [ ]:
# evalutaion logic repetition myself
class EvaluationLogik:
    def StartEndZeitenFlowshop(self, loesung):
        for position, jobid in enumerate(loesung.permutation):
            currentjob = loesung.OutputJobs[jobid]
            if position == 0:
                currentjob.EndTimes = numpy.cumsum(currentjob.ProcessingTimes).tolist()
                currentjob.StartTimes[1:] = currentjob.EndTimes[:-1]
            else:
                currentjob.StartTimes[0] = previousjob.EndTimes[0]
                currentjob.EndTimes[0] = currentjob.StartTimes[0] + currentjob.ProcessingTimes[0]
                for machine in range(1, len(currentjob.ProcessingTimes)):
                    currentjob.StartTimes[machine] = max(previousjob.EndTimes[machine], currentjob.EndTimes[machine-1])
                    currentjob.EndTimes[machine] = currentjob.StartTimes[machine] + currentjob.ProcessingTimes[machine]
            previousjob = currentjob

        loesung.makespan = loesung.OutputJobs[loesung.permutation[-1]].EndTimes[-1]

EvaluationLogik().StartEndZeitenFlowshop(DevilSolution)
print(DevilSolution)
print(DevilSolution.makespan)

The permutation [5, 4, 6, 3, 7, 2, 8, 1, 9, 0, 10] results in a Makespan of 8922
8922


#### c.)
Damit alle Mitarbeiter in der Fertigung detailiert über den Ablaufplan informiert werden können, soll in einer .csv Datei für alle Aufträge aufgelistet werden, wann diese bearbeitet werden sollen. Die Tabelle soll dabei die folgenden Spalten enthalten:

|Machine  |Job      |Start_Setup |End_Setup  |Start    |End 	 |
|---------|---------|------------|-----------|---------|---------|
| 1 	  | 1 	    | 0	         | 0	     | 132 	   | 481 	 |
| 1  	  | 2 	    | 0   	     | 0	     | 0 	   | 132 	 |
| ...	  | ... 	| ...        | ... 	     | ... 	   | ...	 | 

Schreiben Sie eine Methode WriteSolToCsv(), welche eine gegebene Lösung (Instanz der Klasse Solution) in eine csv Datei schreibt. Nutzen Sie dabei das Modul **csv**. Erzeugen Sie anschließend eine Ausgabe von **DevilSolution**. Da die Ausgabe nur mit einem Solution Objekt erfolgen kann, sollten die Methode an die Klasse Solution angehangen werden.



In [6]:
import csv

def WriteSolToCsv(self, fileName):
    with open(fileName, 'w') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Machine', 'Job', 'Start_Setup', 'End_Setup', 'Start_Processing', 'EndProcessing'])
        
        # grouped by jobid
        # for jobid in self.permutation:
        #     job = self.OutputJobs[jobid]
        #     for machine in range(len(job.ProcessingTimes)):

        # grouped by machine
        for machine in range(len(self.OutputJobs[0].ProcessingTimes)):
            for jobid in self.permutation:    # jobs in permutation order
            # for jobid, job in self.OutputJobs.items():  # jobs in ascending order of their IDs
                job = self.OutputJobs[jobid]
                writer.writerow([
                    machine+1,
                    jobid+1,
                    job.StartSetups[machine],
                    job.EndSetups[machine],
                    job.StartTimes[machine],
                    job.EndTimes[machine]
                ])


Solution.WriteSolToCSV = WriteSolToCsv
DevilSolution.WriteSolToCSV('../../../data/Prescriptive_Lessons/DevilSolution.csv')

#### d.)
Nachdem Sie aus der Fertigung hören, dass einige Mitarbeiter Probleme beim Verständnis der csv Ausgabe haben, machen Sie sich auf die Suche nach geeigneten Grafikmodulen. Erfreulicherweise stoßen Sie schnell auf die Methode **timeline()** aus dem Modul **plotly.express**. Zudem hatte scheinbar schon ein anderer Programmierer das gleiche Problem, weshalb Ihnen jetzt ein Python Skript vorliegt, welches Sie direkt zur grafischen Darstellung nutzen können. 

Laden Sie das Skript **Gantt.py** und nutzen Sie die Methode **ganttChart**, um DevilSolution grafisch darzustellen. 

In [ ]:
import Gantt

GanttChart(DevilSolution, '../../../data/Prescriptive_Lessons/DevilSolution_Gantt')

---
### 2. Dispatching Rules zur Erstellung von Lösungen
---
Trotz aller Erfahrung ist sich Traudel Teufel unsicher bezüglich der Qualität ihrer Lösung (DevilSolution). Immerhin gibt es 39916800 mögliche Permutationen für die 11 Aufträge. Aus diesem Grund überlegen Sie, wie sich konstruktiv weitere Lösungen erzeugen lassen, um die Güte der bisherigen Lösung einzuschätzen und eventuell eine bessere Lösung zu finden. 

#### a.)
Um überhaupt ein Gefühl für die Verteilung möglicher Ablaufpläne zu erhalten, lassen sich mit der wohl einfachsten Entscheidungsregel **Random Order of Service** zunächst verschiedene Lösungen erzeugen. Das Vorgehen besteht darin, dass der nächste Auftrag immer zufällig gewählt wird. Schreiben Sie deshalb eine Methode **ROS()**, welche zufällig "x" Fertigungsreihenfolgen erzeugt und die beste zurückgibt. Der Parameter "x" soll dabei der Funktion ebenso wie ein Startwert und die Stammdaten der Aufträge übergeben werden. Bei der Generierung zufälliger Permutationen kann Ihnen das Modul Numpy sicher wieder behilflich sein.

In [13]:
import numpy as np

def ROS(jobList, x, seed):
    np.random.seed(seed)
    num_jobs = len(jobList)
    best_permutation = None
    best_makespan = np.inf
    best_solution = None
    for _ in range(x):
        permutation = np.random.permutation(num_jobs).tolist()
        solution = Solution(jobList, permutation)
        EvaluationLogic().DefineStartEnd(solution)
        if solution.makespan < best_makespan:
            best_makespan = solution.makespan
            best_permutation = permutation
            best_solution = solution
            # print(best_solution)
    return best_solution

ROSSolution = ROS(data.InputJobs,10,2025)
print(ROSSolution)

The permutation [8, 3, 2, 0, 6, 1, 7, 4, 10, 5, 9] results in a Makespan of 8041


Erwarteter Output:

The permutation [ 8  3  2  0  6  1  7  4 10  5  9] results in a Makespan of 8041

In [16]:
# alternative ROS implementation

def ROS1(jobList, x, seed):
    np.random.seed(seed)
    tmpSolution = Solution(jobList, [])
    best_makespan = np.inf

    for _ in range(x):
        tmp_permutation = np.random.permutation(len(jobList)).tolist()
        tmpSolution.setPermutation(tmp_permutation)
        EvaluationLogic().DefineStartEnd(tmpSolution)

        if tmpSolution.makespan < best_makespan:
            best_makespan = tmpSolution.makespan
            best_permutation = tmp_permutation

    best_solution = Solution(jobList, best_permutation)
    EvaluationLogic().DefineStartEnd(best_solution)
    return best_solution

ROSSolution1 = ROS1(data.InputJobs,10,2025)
print(ROSSolution1)

The permutation [8, 3, 2, 0, 6, 1, 7, 4, 10, 5, 9] results in a Makespan of 8041


#### b.)
Ein Mitarbeiter aus der Fertigung weist Sie darauf hin, dass Sie bei 11 Aufträgen auch alle Permutationen berechnen könnten und anschließend die Beste bestimmen können. Sie wollen deshalb eine Funktion **checkAllPermutations()** schreiben, die Ihnen nach Überprüfung aller Reihenfolgen diejenige zurückgibt, die zum besten Makespan führt. Wie lange dauert diese Rechnung?

In [ ]:
from itertools import permutations
import time

def checkAllPermutations(joblist):
    all_permutations = set(permutations(range(len(joblist))))
    best_makespan = np.inf
    tempSolution = Solution(joblist, [])

    for perm in all_permutations:
        tempSolution.setPermutation(perm)
        EvaluationLogic().DefineStartEnd(tempSolution)

        if tempSolution.makespan < best_makespan:
            best_makespan = tempSolution.makespan
            best_permutation = perm
            # print(tempSolution)

    best_solution = Solution(joblist, best_permutation)
    EvaluationLogic().DefineStartEnd(best_solution)
    return best_solution

starttime = time.time()
BruteForceSolution = checkAllPermutations(data.InputJobs)
endtime = time.time()

print(BruteForceSolution)
print(f'Die Brute-Force Methode benötigte {(endtime-starttime)/60:.2f} Minuten.\n')
print(f'Die Beste gefundene Permutation ist {BruteForceSolution.permutation}\n')
print(f'Dies führt zu einem Makespan von {BruteForceSolution.makespan}\n')

The permutation (7, 4, 2, 10, 3, 1, 6, 8, 5, 9, 0) results in a Makespan of 7038
Die Brute-Force Methe benötigte 12.61 Minuten.

Die Beste gefundene Permutation ist (7, 4, 2, 10, 3, 1, 6, 8, 5, 9, 0)

Dies führt zu einem Makespan von 7038



In [17]:
BruteForceSolution.WriteSolToCSV('../../../data/Prescriptive_Lessons/BruteForceSolution.csv')

In [ ]:
from itertools import permutations
import time
# Define Method
def CheckAllPermutations(jobList):
    allPerms = set(permutations(range(len(jobList))))
    bestCmax = numpy.inf
    tmpSolution = Solution(jobList,0)
    for tmpPerm in allPerms:
        tmpSolution.SetPermutation(tmpPerm)
        EvaluationLogic().DefineStartEnd(tmpSolution)       
        if(tmpSolution.Makespan < bestCmax):
            bestCmax = tmpSolution.Makespan
            bestPerm = tmpPerm
    bestSol = Solution(jobList,bestPerm)
    EvaluationLogic().DefineStartEnd(bestSol)
    return bestSol       

# Use function to get best Solution
time1 = time.time() 
bestSolution = CheckAllPermutations(data.InputJobs)
time2 = time.time() 
# Print Results
print("Rechenzeit beträgt "+ str((time2-time1)/60) + " Minuten")
print("Beste Lösung mit Cmax "+ str(bestSolution.Cmax))
print("Beste Reihenfolge ist: ", end="")
for x in bestSolution.Permutation:
    print(x,end=' ')

Erwarteter Output:

    Rechenzeit beträgt 111.684 Minuten <br>
    Beste Lösung mit Makespan 7038 <br>
    Beste Reihenfolge ist: 7 2 0 4 6 10 3 5 8 1 9

#### c.)
Nachdem eine vollständige Enumeration selbst bei kleinen Problemen langfristig keine Alternative darstellt, schlagen Sie vor, das Problem mit statischen Einplanungsregeln zu lösen. Frau Teufel ist von dieser Idee begeistert und möchte gern, dass Sie die folgenden Regeln implementieren: <br>
* FCFS (First Come First Serve)
* SPT (Shortest Processing Time)
* LPT (Longest Processing Time) 

Schreiben Sie für jede dieser Regeln eine Methode. Die Funktionen sollen jeweils ein Objekt der Klasse **Solution** ausgeben.



In [48]:
class Heuristics:
    def FCFS(self, jobList):
        permutation = list(range(len(jobList)))
        solution = Solution(jobList, permutation)
        EvaluationLogic().DefineStartEnd(solution)
        return solution
    
    def SPT(self, jobList, CumulatedMachines=False):
        if CumulatedMachines:
            ProcessingTimes = {jobid: sum(job.ProcessingTimes) for jobid, job in enumerate(jobList)}
        else:
            ProcessingTimes = {jobid: job.ProcessingTimes[0] for jobid, job in enumerate(jobList)}
        
        # sorting dict by processing time values and extracting only the job ids (i.e. keys)
        permutation = [jobid for jobid, _ in sorted(ProcessingTimes.items(), key= lambda x: x[1])]
        solution = Solution(jobList, permutation)
        EvaluationLogic().DefineStartEnd(solution)
        return solution
    
    def LPT(self, jobList, CumulatedMachines = False):
        if CumulatedMachines:
            ProcessingTimes = {jobid: sum(job.ProcessingTimes) for jobid, job in enumerate(jobList)}
        else:
            ProcessingTimes = {jobid: job.ProcessingTimes[0] for jobid, job in enumerate(jobList)}

        permutation = [job[0] for job in sorted(ProcessingTimes.items(), key=lambda x:x[1], reverse=True)]
        solution = Solution(jobList, permutation)
        EvaluationLogic().DefineStartEnd(solution)
        return solution
    
    def EDD(self, jobList):
        dueDates = {jobid: job.DueDate for jobid, job in enumerate(jobList)}
        permutation = [jobid for jobid, duedate in sorted(dueDates.items(), key=lambda x: x[1])]
        solution = Solution(jobList, permutation)
        EvaluationLogic().DefineStartEnd(solution)
        return solution
    
    def ROS(self, jobList, x, seed):
        np.random.seed(seed)
        tmpSolution = Solution(jobList, [])
        best_makespan = np.inf

        for _ in range(x):
            tmp_permutation = np.random.permutation(len(jobList)).tolist()
            tmpSolution.setPermutation(tmp_permutation)
            EvaluationLogic().DefineStartEnd(tmpSolution)
            if tmpSolution.makespan < best_makespan:
                best_makespan = tmpSolution.makespan
                best_permutation = tmp_permutation
        
        best_solution = Solution(jobList, best_permutation)
        EvaluationLogic().DefineStartEnd(best_solution)
        return best_solution

##### Testaufrufe für Verständnis

In [21]:
sum(data.InputJobs[1].ProcessingTimes)

2518

In [30]:
test_dict = {0: 45, 1: 30, 2: 60, 3: 75, 4: 90, 5: 50, 6: 120, 7: 99, 8: 150, 9: 165, 10: 70}
# list comprehension
[jobid for jobid, processtime in sorted(test_dict.items(), key=lambda x: x[1])]

# nacheinander
test_dict_sorted = sorted(test_dict.items(), key=lambda x:x[1])
test_dict_sorted
[jobid for jobid, processtime in test_dict_sorted]

[1, 0, 5, 2, 10, 3, 4, 7, 6, 8, 9]

#### Lösungen der Konstruktiven Heuristiken

In [46]:
FCFS_Sol = Heuristics().FCFS(data.InputJobs)
print(FCFS_Sol)

The permutation [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] results in a Makespan of 9298


Erwarteter Output:

    The permutation [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] results in a Makespan of 9298

In [41]:
SPTSol = Heuristics().SPT(data.InputJobs)
print(SPTSol)

SPTSol1 = Heuristics().SPT(data.InputJobs, True)
print(SPTSol1)

The permutation [2, 7, 8, 0, 3, 4, 6, 10, 1, 5, 9] results in a Makespan of 7718
The permutation [0, 7, 3, 5, 8, 2, 1, 9, 4, 6, 10] results in a Makespan of 8848


Erwarteter Output:

    The permutation [2, 7, 8, 0, 3, 4, 6, 10, 1, 5, 9] results in a Makespan of 7718

In [44]:
LPTSol = Heuristics().LPT(data.InputJobs)
print(LPTSol)

LPTSol1 = Heuristics().LPT(data.InputJobs, True)
print(LPTSol1)

The permutation [9, 5, 1, 6, 10, 4, 3, 0, 8, 7, 2] results in a Makespan of 10649
The permutation [10, 6, 4, 9, 1, 2, 8, 5, 3, 7, 0] results in a Makespan of 9101


Erwarteter Output:

    The permutation [9, 5, 1, 6, 10, 4, 3, 0, 8, 7, 2] results in a Makespan of 10649

In [47]:
EDDSol = Heuristics().EDD(data.InputJobs)
print(EDDSol)

The permutation [0, 7, 3, 5, 8, 2, 1, 9, 4, 6, 10] results in a Makespan of 8848


In [54]:
ROSSol = Heuristics().ROS(data.InputJobs, 10000, 2025)
print(ROSSol)

The permutation [7, 0, 2, 10, 8, 4, 3, 9, 6, 5, 1] results in a Makespan of 7038


#### d.) NEH Heuristik
Auch wenn Sie durch die Dispatching Rules sehr einfach und schnell Lösungen generieren können, sind Sie doch von der Lösungsgüte etwas enttäuscht. Selbst die SPT Regel führt zu fast 10% Abweichung von der optimalen Lösung. Aus diesem Grund erachten Sie es als sinnvoll, aufwändigere konstruktive Verfahren zu analysieren. Schnell stellen Sie fest, dass die NEH Heuristik zu den am meisten verwendeten Ansätzen gehört und häufig sehr gute Startlösungen liefert. Deshalb wollen Sie im Folgenden selbst eine Methode **NEH()** schreiben und das Verfahren implementieren.

In [ ]:
# Eigener Ansatz mti Einsetzen der Jobs an verschiedenen Positionen
from copy import deepcopy

def NEH(self, jobList):
    sum_ProcessingTimes = {jobid: sum(job.ProcessingTimes) for jobid, job in enumerate(jobList)}
    planning_order = [job[0] for job in sorted(sum_ProcessingTimes.items(), key=lambda x: x[1], reverse=True)]
    partial_permutation = [planning_order[0]]

    for jobid in planning_order[1:]:
        best_makespan = np.inf
        iteration_permutation = None
        tmp_solution = Solution(jobList, [])
        for position in range(len(partial_permutation)+1):
            tmp_permutation = deepcopy(partial_permutation)
            tmp_permutation.insert(position, jobid)
            tmp_solution.setPermutation(tmp_permutation)
            # tmp_solution = Solution(jobList, tmp_permutation)
            EvaluationLogic().DefineStartEnd(tmp_solution)
            if tmp_solution.makespan < best_makespan:
                best_makespan = tmp_solution.makespan
                iteration_permutation = tmp_permutation
        partial_permutation = iteration_permutation

    best_solution = Solution(jobList, partial_permutation)
    EvaluationLogic().DefineStartEnd(best_solution)
    return best_solution

Heuristics.NEH = NEH

In [60]:
NEHSol = Heuristics().NEH(data.InputJobs)
print(NEHSol)

The permutation [7, 0, 4, 8, 2, 10, 3, 6, 5, 1, 9] results in a Makespan of 7038


In [62]:
NEHSol.total_tardiness

-1

#### Musterlösung (mit Swaps)
Performanter hinsichtlich Geschwindigkeit und Arbeitsspeicher, da nur eine Permutation generiert und anschließend Positionstausche erfolgen. Bei meiner eigenen Lösung wird hingegen für jede Position eine neue Permutation generiert, was hardwareseitig fordernder ist.

In [ ]:
# Musterlsung mit Tauschoperation anstelle von separater Einfügeoperation
from copy import deepcopy

def DetermineBestInsertion(solution, jobToInsert):
    ###
    # insert job at front of permutation
    solution.Permutation.insert(0, jobToInsert)
    bestPermutation = deepcopy(solution.Permutation)
    
    EvaluationLogic().DefineStartEnd(solution)
    bestCmax = solution.Makespan

    ###
    # swap job i to each position and check for improvement
    lengthPermutation = len(solution.Permutation) - 1
    for j in range(0, lengthPermutation):
        solution.Permutation[j], solution.Permutation[j + 1] = solution.Permutation[j+1], solution.Permutation[j]
        EvaluationLogic().DefineStartEnd(solution)
        if(solution.Makespan < bestCmax):
            bestCmax = solution.Makespan
            bestPermutation = [x for x in solution.Permutation]

    solution.Makespan = bestCmax
    solution.Permutation = bestPermutation

def NEH(jobList):
    jobPool = []
    tmpPerm = []
    bestCmax = 0
    # Calculate sum of processing times and sort
    for i in range(len(jobList)):
        jobPool.append((i,sum(jobList[i].ProcessingTime(x) for x in range(len(jobList[i].Operations)))))
    jobPool.sort(key=lambda x: x[1], reverse=True)

    # Initalize input
    tmpNEHOrder = [x[0] for x in jobPool]
    tmpPerm.append(tmpNEHOrder[0])
    tmpSolution = Solution(jobList,tmpPerm)

    # Add next jobs in a loop and check all permutations
    for i in range(1,len(tmpNEHOrder)):
        # add next job to end and calculate makespan
        DetermineBestInsertion(tmpSolution, tmpNEHOrder[i])
    
    return tmpSolution

NEHSol = NEH(data.InputJobs)    
print(NEHSol)



Erwarteter Output:

    The permutation [7, 0, 4, 8, 2, 10, 3, 6, 5, 1, 9] results in a Makespan of 7038

#### e.)

Glücklich darüber, dass Sie mit Hilfe der NEH eine sehr gute Lösung gefunden haben. Wollen Sie Ihre implementierten konstruktiven Lösungsansätze abschließend in einer eigenen Klasse zusammenfassen. Nennen Sie die neue Klasse **ConstructiveHeuristic**. Damit Sie zukünftig einfach auf die einzelnen Methoden zugreifen können, entscheiden Sie sich, eine Hauptroutine zu schreiben, über welche Sie unter Angabe der jeweiligen konstruktiven Heuristik auf die Methoden zugreifen können.

In [ ]:
# habe bereits Klasse erstellt. Wenn zuvor jedoch einzelne Funktionen definiert wurden,
# sieht Musterlösung wie folgt aus:
class ConstructiveHeuristics:
    def Run(inputData, solutionMethod):
        if solutionMethod == 'FCFS':
            solution = FirstComeFirstServe(inputData.InputJobs)
        elif solutionMethod == 'SPT':
            solution = ShortestProcessingTime(inputData.InputJobs)
        elif solutionMethod == 'LPT':
            solution = LongestProcessingTime(inputData.InputJobs)
        elif solutionMethod == 'ROS':
            solution = ROS(inputData.InputJobs, 323, 10)
        elif solutionMethod == 'NEH':
            solution = NEH(inputData.InputJobs)
        else:
            print('Unkown constructive solution method.')    
        return solution

print(ConstructiveHeuristics().Run(data,"NEH"))

---
### 3. Liefertreue 
--- 
Ihre bisherigen Lösungsversuchen konzentrieren sich auf eine möglichst effiziente Fertigung, indem die Kapazitäten möglichst gut ausgelastet werden. Allerdings stellt für Dr. Best auch die Liefertreue ein wichtiges Kriterium dar, damit die Kunden immer pünklich ihre Zähne putzen können. Deshalb soll im Folgenden **Total Tardiness** als alternatives Zielkriterium betrachtet werden.

a) 
Welche gesamten Verspätungen ergeben sich mit den bisherigen konstruktiven Lösungsansätzen **NEH**, **SPT** und **FCFS**? Schreiben Sie zuvor zwei Methoden in der Klasse EvaluationLogic, welche Total Tardiness und Total Weighted Tardiness berechnen.

In [63]:
def CalculateTotalTardiness(self, solution):
    solution.total_tardiness = 0
    for jobid in solution.permutation:
        job = solution.OutputJobs[jobid]
        job.Tardiness = max(0, job.EndTimes[-1] - job.DueDate)
        solution.total_tardiness += job.Tardiness

def CalculateTotalWeightedTardiness(self, solution):
    solution.total_weighted_tardiness = 0
    for jobid in solution.permutation:
        job = solution.OutputJobs[jobid]
        job.WeightedTardiness = job.TardCost * max(0, job.EndTimes[-1] - job.DueDate)
        solution.total_weighted_tardiness += job.WeightedTardiness

EvaluationLogic.CalculateTotalTardiness = CalculateTotalTardiness
EvaluationLogic.CalculateTotalWeightedTardiness = CalculateTotalWeightedTardiness

In [66]:
EvaluationLogic().CalculateTotalTardiness(NEHSol)
EvaluationLogic().CalculateTotalWeightedTardiness(NEHSol)

EvaluationLogic().CalculateTotalTardiness(FCFS_Sol)
EvaluationLogic().CalculateTotalWeightedTardiness(FCFS_Sol)

EvaluationLogic().CalculateTotalTardiness(LPTSol)
EvaluationLogic().CalculateTotalWeightedTardiness(LPTSol)

EvaluationLogic().CalculateTotalTardiness(EDDSol)
EvaluationLogic().CalculateTotalWeightedTardiness(EDDSol)

EvaluationLogic().CalculateTotalTardiness(SPTSol)
EvaluationLogic().CalculateTotalWeightedTardiness(SPTSol)

print(
    f'Total Tardiness of NEH Solution: {NEHSol.total_tardiness}\n'
    f'Total Tardiness of FCFS Solution: {FCFS_Sol.total_tardiness}\n'
    f'Total Tardiness of LPT Solution: {LPTSol.total_tardiness}\n'
    f'Total Tardiness of EDD Solution: {EDDSol.total_tardiness}\n'
    f'Total Tardiness of SPT Solution: {SPTSol.total_tardiness}\n'
)

print(
    f'Total Weighted Tardiness of NEH Solution: {NEHSol.total_weighted_tardiness}\n'
    f'Total Weighted Tardiness of FCFS Solution: {FCFS_Sol.total_weighted_tardiness}\n'
    f'Total Weighted Tardiness of LPT Solution: {LPTSol.total_weighted_tardiness}\n'
    f'Total Weighted Tardiness of EDD Solution: {EDDSol.total_weighted_tardiness}\n'
    f'Total Weighted Tardiness of SPT Solution: {SPTSol.total_weighted_tardiness}'
)

Total Tardiness of NEH Solution: 16329
Total Tardiness of FCFS Solution: 26812
Total Tardiness of LPT Solution: 40081
Total Tardiness of EDD Solution: 15859
Total Tardiness of SPT Solution: 21866

Total Weighted Tardiness of NEH Solution: 3265800
Total Weighted Tardiness of FCFS Solution: 5362400
Total Weighted Tardiness of LPT Solution: 8016200
Total Weighted Tardiness of EDD Solution: 3171800
Total Weighted Tardiness of SPT Solution: 4373200


In [ ]:
def CalculateTardiness(self, currentSolution):
        totalTardiness = 0
        for key in currentSolution.OutputJobs:
            if(currentSolution.OutputJobs[key].EndTimes[-1] - currentSolution.OutputJobs[key].DueDate > 0):
                currentSolution.OutputJobs[key].Tardiness = currentSolution.OutputJobs[key].EndTimes[-1] - currentSolution.OutputJobs[key].DueDate
                totalTardiness += currentSolution.OutputJobs[key].EndTimes[-1] - currentSolution.OutputJobs[key].DueDate
        currentSolution.TotalTardiness = totalTardiness       

setattr(EvaluationLogic,"CalculateTardiness",CalculateTardiness)

def CalculateWeightedTardiness(self, currentSolution):
    totalWeightedTardiness = 0
    for key in currentSolution.OutputJobs:
        if(currentSolution.OutputJobs[key].EndTimes[-1] - currentSolution.OutputJobs[key].DueDate > 0):
            currentSolution.OutputJobs[key].Tardiness = currentSolution.OutputJobs[key].EndTimes[-1] - currentSolution.OutputJobs[key].DueDate
            totalWeightedTardiness += (currentSolution.OutputJobs[key].EndTimes[-1] - currentSolution.OutputJobs[key].DueDate) * currentSolution.OutputJobs[key].TardCost
    currentSolution.TotalWeightedTardiness = totalWeightedTardiness

setattr(EvaluationLogic,"CalculateWeightedTardiness",CalculateWeightedTardiness)


EvaluationLogic().CalculateTardiness(NEHSol)
print("NEH Regel führt zu gesamten Verspätungen von: "+ str(NEHSol.TotalTardiness))
EvaluationLogic().CalculateTardiness(FCFSSol)
print("FCFS Regel führt zu gesamten Verspätungen von: "+ str(FCFSSol.TotalTardiness))
EvaluationLogic().CalculateTardiness(SPTSol)
print("SPT Regel führt zu gesamten Verspätungen von: "+ str(SPTSol.TotalTardiness))

Erwarteter Output:

    NEH Regel führt zu gesamten Verspätungen von: 18152
    FCFS Regel führt zu gesamten Verspätungen von: 26812
    SPT Regel führt zu gesamten Verspätungen von: 21866

#### b)
Da die bisherigen konstruktiven Lösungsansätze noch nicht auf Tardiness ausgerichtet sind, vermuten Sie, dass noch Potential zur Verbesserung besteht. Deshalb wollen Sie im Folgenden die Entscheidungsregel **Earliest Due Date** zur Lösungserzeugung nutzen. Schreiben Sie dafür eine entsprechende Funktion. Wie hoch sind hierbei die Verspätungen?

In [70]:
# siehe oben
print(EDDSol)
print(f'EDD Regel führt zu gesamten Verspätungen von {EDDSol.total_tardiness}')

The permutation [0, 7, 3, 5, 8, 2, 1, 9, 4, 6, 10] results in a Makespan of 8848
EDD Regel führt zu gesamten Verspätungen von 15859


Erwarteter Output:
   
    EDD Regel führt zu gesamten Verspätungen von: 15859